# 01 — Dataset Exploration

This notebook explores the dataset structure, class distribution, bounding box statistics, and sample visualizations.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2

from src.preprocessing.dataset_utils import (
    load_yolo_labels,
    validate_dataset,
    get_dataset_stats,
    yolo_to_xyxy,
    generate_sample_dataset,
)
from src.utils.visualization import plot_class_distribution, generate_color_palette

## Generate Sample Dataset (Optional)

If you don't have a real dataset yet, generate a synthetic one for testing.

In [ ]:
# Uncomment to generate a synthetic dataset for testing
# generate_sample_dataset(Path('../data/dataset'), num_images=100)

## Dataset Configuration

In [ ]:
DATASET_DIR = Path('../data/dataset')
CLASS_NAMES = {0: 'person', 1: 'car', 2: 'bicycle', 3: 'motorcycle'}

splits = ['train', 'val', 'test']
for split in splits:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    n_images = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_labels = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split:>5}: {n_images} images, {n_labels} labels')

## Class Distribution

In [ ]:
class_counts = {i: 0 for i in range(len(CLASS_NAMES))}

for split in splits:
    lbl_dir = DATASET_DIR / split / 'labels'
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        labels = load_yolo_labels(lbl_file)
        for lbl in labels:
            cid = lbl['class_id']
            if cid in class_counts:
                class_counts[cid] += 1

print('Class distribution:')
for cid, count in class_counts.items():
    print(f'  {CLASS_NAMES[cid]}: {count}')

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(8, 5))
names = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]
ax.bar(names, counts, color=plt.cm.Set2.colors[:len(names)])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Bounding Box Analysis

In [ ]:
widths, heights, centers_x, centers_y = [], [], [], []

for split in splits:
    lbl_dir = DATASET_DIR / split / 'labels'
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        labels = load_yolo_labels(lbl_file)
        for lbl in labels:
            widths.append(lbl['width'])
            heights.append(lbl['height'])
            centers_x.append(lbl['x_center'])
            centers_y.append(lbl['y_center'])

if widths:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(widths, bins=30, alpha=0.7, label='Width')
    axes[0].hist(heights, bins=30, alpha=0.7, label='Height')
    axes[0].set_title('Bounding Box Size Distribution')
    axes[0].legend()

    axes[1].scatter(centers_x, centers_y, alpha=0.3, s=5)
    axes[1].set_title('Box Center Positions')
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].invert_yaxis()

    axes[2].scatter(widths, heights, alpha=0.3, s=5)
    axes[2].set_title('Width vs Height')
    axes[2].set_xlabel('Width')
    axes[2].set_ylabel('Height')

    plt.tight_layout()
    plt.show()
else:
    print('No bounding boxes found — generate or add a dataset first.')

## Sample Image Visualization

In [ ]:
train_img_dir = DATASET_DIR / 'train' / 'images'
train_lbl_dir = DATASET_DIR / 'train' / 'labels'

image_files = sorted(train_img_dir.glob('*.[jp][pn]g')) if train_img_dir.exists() else []

if image_files:
    colors = generate_color_palette(len(CLASS_NAMES))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flat, image_files[:6]):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = train_lbl_dir / img_path.with_suffix('.txt').name
        if lbl_path.exists():
            labels = load_yolo_labels(lbl_path)
            for lbl in labels:
                x1, y1, x2, y2 = yolo_to_xyxy(lbl, w, h)
                cid = lbl['class_id']
                color = [c / 255 for c in colors[cid % len(colors)]]
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, CLASS_NAMES.get(cid, str(cid)),
                        color=color, fontsize=8, weight='bold')
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No training images found.')